# Gates, Interference and Measurement Bases
Chapter 5 companion. Predict before running each calculation. All states and probabilities below are ideal simulator predictions. Each sampled shot represents a fresh preparation; simulator statevectors are not extra physical readout data. Angles in code are radians.


In [ ]:
%pip -q install "qiskit>=2,<3" "numpy>=2,<3" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
zero = np.array([1, 0], dtype=complex)
one = np.array([0, 1], dtype=complex)
H = np.array([[1,1],[1,-1]], dtype=complex)/np.sqrt(2)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.diag([1,-1]).astype(complex)
S = np.diag([1,1j])
T = np.diag([1,np.exp(1j*np.pi/4)])
plus = H @ zero
minus = H @ one
plus_i = S @ plus


## 1. Unitarity and inverse evolution
For each gate, compare an independent matrix with Qiskit and check its action on a complex input.


In [ ]:
psi = np.array([np.sqrt(3)/2, 1j/2])
for name, U in {"x":X,"y":Y,"z":Z,"h":H,"s":S,"t":T}.items():
    qc = QuantumCircuit(1)
    getattr(qc, name)(0)
    assert np.allclose(Operator(qc).data, U)
    assert np.allclose(U.conj().T @ U, np.eye(2))
    transformed = Statevector(psi).evolve(qc)
    assert np.allclose(transformed.data, U @ psi)
    assert np.allclose(U.conj().T @ transformed.data, psi)
    print(name.upper(), transformed.data, transformed.probabilities())


## 2. Hadamard and an inserted measurement
Without a measurement H²=I. With an intermediate Z measurement whose record is ignored, calculate conditional branches and average their final probabilities. No single pure vector is assigned to that ensemble.


In [ ]:
assert np.allclose(H @ H, np.eye(2))
coherent = H @ H @ zero
branch_weights = np.abs(H @ zero)**2
measured_probabilities = sum(w * np.abs(H @ b)**2 for w,b in zip(branch_weights,[zero,one]))
print("Uninterrupted H-H:", np.abs(coherent)**2)
print("Intermediate measurement, ignored record:", measured_probabilities)
assert np.allclose(measured_probabilities, [0.5,0.5])


## 3. The interference fringe
Predict the results at φ=0, π/2 and π. Compare the analytic formula, NumPy matrices and Qiskit.


In [ ]:
phases = np.linspace(0, 2*np.pi, 121)
p0 = []
for phi in phases:
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.p(phi,0)
    qc.h(0)
    sv = Statevector.from_instruction(qc)
    expected = np.abs(H @ np.diag([1,np.exp(1j*phi)]) @ H @ zero)**2
    assert np.allclose(sv.probabilities(), expected)
    assert np.allclose(expected[0], (1+np.cos(phi))/2)
    p0.append(expected[0])
plt.plot(phases,p0,label="Coherent P(0)")
plt.axhline(.5,color="grey",linestyle="--",label="Intermediate measurement, record ignored")
plt.xlabel("Phase φ (radians)")
plt.ylabel("Probability of final zero")
plt.ylim(-.03,1.03)
plt.legend()
plt.show()


## 4. Phase powers and rotations
Check exact matrix equalities separately from equivalence up to global phase.


In [ ]:
assert np.allclose(T@T,S)
assert np.allclose(np.linalg.matrix_power(T,4),Z)
assert np.allclose(np.linalg.matrix_power(T,8),np.eye(2))
for theta in [-np.pi,0,np.pi/2,2*np.pi/3,2*np.pi]:
    c,s = np.cos(theta/2),np.sin(theta/2)
    independent = {"rx":np.array([[c,-1j*s],[-1j*s,c]]), "ry":np.array([[c,-s],[s,c]]), "rz":np.diag([np.exp(-1j*theta/2),np.exp(1j*theta/2)])}
    for name,U in independent.items():
        qc=QuantumCircuit(1)
        getattr(qc,name)(theta,0)
        assert np.allclose(Operator(qc).data,U)
qc=QuantumCircuit(1)
qc.ry(2*np.pi/3,0)
assert np.allclose(Statevector.from_instruction(qc).probabilities(),[.25,.75])
qc=QuantumCircuit(1)
qc.rz(np.pi/3,0)
rz_psi=Statevector(psi).evolve(qc)
p_psi=Statevector(np.diag([1,np.exp(1j*np.pi/3)]) @ psi)
assert rz_psi.equiv(p_psi)
assert not np.allclose(rz_psi.data,p_psi.data)


## 5. Circuit order
The time order H, S, H, T is the matrix THSH. Inspect every intermediate state.


In [ ]:
qc=QuantumCircuit(1)
v=zero.copy()
for name,U in [("h",H),("s",S),("h",H),("t",T)]:
    getattr(qc,name)(0)
    v=U@v
    print(name,v)
assert np.allclose(v,T@H@S@H@zero)
assert np.allclose(v,Statevector.from_instruction(qc).data)
print(qc.draw(output="text"))
assert np.allclose(v,[(1+1j)/2,1/np.sqrt(2)])


## 6. Choosing readout and sampling fresh preparations
X readout uses H before Z; Y readout uses S† then H. Record 0 is positive and record 1 negative. Statevector.sample_counts samples the stored ideal distribution without collapsing the stored state.


In [ ]:
def readout_state(v,basis):
    qc=QuantumCircuit(1)
    if basis=="X":
        qc.h(0)
    elif basis=="Y":
        qc.sdg(0)
        qc.h(0)
    elif basis!="Z":
        raise ValueError("Choose X, Y or Z")
    return Statevector(v).evolve(qc)
for basis,positive in [("X",plus),("Y",plus_i)]:
    assert np.allclose(readout_state(positive,basis).probabilities(),[1,0])
for v in [psi,plus,minus,plus_i]:
    for basis,positive in [("X",plus),("Y",plus_i),("Z",zero)]:
        prediction=abs(np.vdot(positive,v))**2
        assert np.allclose(readout_state(v,basis).probabilities()[0],prediction)
readout=readout_state(psi,"Y")
saved=readout.data.copy()
readout.seed(2026)
print("Exact Y probabilities:",readout.probabilities())
print("1000 fresh-shot records:",readout.sample_counts(1000))
assert np.allclose(readout.data,saved)


## Practice
1. Change the interference experiment to φ=π/3. Predict 200-shot expected counts before sampling.
2. Compare H→Z and Z→H on zero using X readout.
3. Prepare plus-i, implement Y readout, and explain why the retained detector-frame state differs from the ideal input-frame conditional state.
4. Replace Ry(θ) with Rx(θ). Compare complex amplitudes and Z probabilities.

Reference: [Qiskit QuantumCircuit API](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.3/qiskit.circuit.QuantumCircuit).
